# Distance-based Fryer, transit diagnostics, and NYC PSSI maps

This notebook reads saved local results. It makes no downloads or Routes API calls.

The new Fryer analysis uses **1 km geographic neighbors on populated 2020 Census blocks**, with companion tract results and 0.5/1.5 km sensitivity. It covers the 11 comparison cities plus NYC. The existing PSSI transit comparison and NYC 70-section model remain unchanged.

The [paper](https://eml.berkeley.edu/~fechenique/published/echenique-fryer-qje.pdf) and [authors' code/data](https://eml.berkeley.edu/~fechenique/segregation/) are documented in `data/references/fryer/citations.json`.


In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import sparse

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'scripts/distance_fryer.py').exists())
BASE = ROOT / 'outputs/distance_fryer'
block_scores = pd.read_csv(BASE / 'block_scores_1km.csv')
tract_scores = pd.read_csv(BASE / 'tract_scores_1km.csv')
sensitivity = pd.read_csv(BASE / 'scores_all_radii.csv')
comparison = pd.read_csv(BASE / 'pssi_and_distance_fryer.csv')
verification = json.loads((BASE / 'verification.json').read_text())
print('Block city/category scores:', len(block_scores), 'All radii/scales:', len(sensitivity))
print('Independent verification:', verification['passed'])
print('Full report:', BASE / 'index.html')


## Definition and interpretation

For different nodes $i,j$, let $W_{ij}=1$ if their representative points are within the distance threshold and zero otherwise. Set $W_{ii}=0$. Let $d_i=\sum_jW_{ij}$, counting neighbors of every category. For $d_i>0$, $R_{ij}=W_{ij}/d_i$.

Each node receives its largest of the four category counts. Ties prefer Hispanic, Asian, Black, then White. This explicit rule matches the authors' sampled published labels, including labels below 50%; the original preprocessing script is unavailable. All-four-zero nodes remain other/unclassified. Resident counts overlap, but assigned node labels do not.

Select nodes assigned to a category and having at least one geographic neighbor. Restrict $R$ without renormalization. For each same-category component $c$, compute its dominant eigenvalue $\lambda_c$ and right eigenvector $v_c$ with sum one:

$$SSI=\sum_c\frac{|c|}{N_r^{eligible}}\lambda_c,\qquad s_i=|c|\lambda_c v_{c,i}.$$

Geographic isolates are excluded from the denominator, matching author code. Eligible nodes with only other-category neighbors remain in the denominator with local score zero. No eligible nodes means undefined. Local SSI can exceed one.

For **all city residents**, $K=N_r/N$. Requested adjustment is $SSI-K$; normalization is $(SSI-K)/(1-K)$. The paper subtracts a simulated null baseline, so the $K$ subtraction is explicitly labeled a population-share adjustment. Counts of residents living in nodes assigned to another category still contribute to $K$.


In [ ]:
columns = ['city', 'group', 'population_share_K', 'fryer_ssi', 'fryer_ssi_minus_K',
           'fryer_ssi_normalized', 'eligible_assigned_node_count', 'assigned_degree_zero_excluded', 'status']
with pd.option_context('display.max_rows', None, 'display.max_columns', None):
    display(block_scores[columns])
    display(pd.read_csv(BASE / 'column_dictionary.csv'))


In [ ]:
for name in ['block_fryer_1km.png', 'tract_fryer_1km.png', 'radius_sensitivity.png']:
    fig, ax = plt.subplots(figsize=(17, 10))
    ax.imshow(plt.imread(BASE / name)); ax.axis('off')
    plt.tight_layout(); plt.show()


## Compare with PSSI

This table has the original 11 cities. PSSI uses the saved individual-tract walking/transit model; the two new Fryer versions use blocks and tracts with distance-only neighbors. NYC's existing PSSI uses 70 sections and stays separate. Different node scales and group representations are not interchangeable.


In [ ]:
with pd.option_context('display.max_columns', None):
    display(comparison)
    display(pd.read_csv(BASE / 'comparison_column_dictionary.csv'))


## Inspect a distance-Fryer component

Node IDs and matrix index maps preserve the exact order. Changing `city_id`, `scale`, and `group` selects a different case. These local scores use paper-scaled component eigenvectors, not the PSSI right vector.


In [ ]:
city_id, scale, group = 'new_york_city', 'block', 'black_alone'
folder = BASE / city_id / scale
nodes = pd.read_csv(folder / 'nodes.csv', dtype={'node_id': str})
local = pd.read_csv(folder / 'local_scores_1km.csv.gz', dtype={'node_id': str})
selected = local[local.group.eq(group) & local.eligible_assigned]
value = block_scores[block_scores.city_id.eq(city_id) & block_scores.group.eq(group)].fryer_ssi.iloc[0]
np.testing.assert_allclose(selected.local_fryer_ssi.mean(), value, atol=1e-10)
print('Mean eligible local SSI = city SSI:', value)
display(selected.sort_values('local_fryer_ssi', ascending=False).head(15))
R = sparse.load_npz(folder / 'neighbor_weights_1km.npz')
B = sparse.load_npz(folder / f'B_1km_{group}.npz')
idx = np.load(folder / f'B_node_indices_{group}.npy')
assert (B - R[idx][:, idx]).nnz == 0
print('R shape:', R.shape, 'Induced B shape:', B.shape)


## Houston and Phoenix: why some PSSI scores barely change

The saved feeds include bus and rail in both cities. Transit lowers costs for approximately 45% of Houston's pairs and 65% of Phoenix's, with large Black/Hispanic score changes. White/Asian spectral scores can be nearly fixed by a subnetwork with no modeled boarding access. That unchanged principal matrix supplies a lower bound close to the whole-city dominant eigenvalue.

Most zero-access cases come from the modeled walking-access radii (1 km bus, 3 km rail, after the 1.3 distance factor). In Houston this excludes several influential tracts near Kingwood/Cypress park-and-ride stops. Phoenix's strongest White driver is much farther from active stops. This does not establish absence of actual transit journeys. Expected-time schedules, access cutoffs, the selected morning service window and conservative water checks all limit routing accuracy.


In [ ]:
DIAG = ROOT / 'outputs/multicity/transit_diagnostics'
diagnostics = pd.read_csv(DIAG / 'category_diagnostics.csv')
display(diagnostics[['city_id', 'group', 'segregation_index_normalized_with_transit',
                     'segregation_index_normalized_walking_only',
                     'unserved_eigenvalue_fraction_of_city_with_transit',
                     'network_isolation_with_transit', 'network_isolation_walking_only']])
display(pd.read_csv(DIAG / 'city_coverage.csv'))
print('Detailed diagnostic report:', DIAG / 'index.html')


## NYC: four PSSI eigenvector maps

Maps use the existing **70-section NYC walking/subway model**, including its documented missing-service transit proxies. They show $70v_i$: section mean one, where the saved right eigenvector satisfies $Av=\lambda v$ and $\sum_i v_i=1$. The shared logarithmic color scale makes relative patterns comparable. These colors are neither citywide normalized PSSI nor distance-Fryer scores. The original sum-one vector is preserved in the CSV.


In [ ]:
MAPS = ROOT / 'outputs/nyc_routes/eigenvector_maps'
fig, ax = plt.subplots(figsize=(14, 15))
ax.imshow(plt.imread(MAPS / 'nyc_four_category_pssi_eigenvectors.png')); ax.axis('off')
plt.tight_layout(); plt.show()
section_scores = pd.read_csv(MAPS / 'nyc_section_eigenvectors.csv')
np.testing.assert_allclose(section_scores.groupby('category').pssi_right_eigenvector_v.sum(), 1, atol=1e-12)
display(section_scores)
print('Four separate high-resolution plots:', MAPS / 'index.html')


## Validation and reproduction

All computations reuse saved data. The primary run disabled socket connections. The city and local matrices, population adjustments, component aggregation, and NYC eigenvector ordering were verified independently. Tract-based results with no eligible assigned nodes are intentionally undefined; no zero is fabricated.

```bash
MPLCONFIGDIR=/private/tmp/apma4903-matplotlib .venv/bin/python -m scripts.distance_fryer
.venv/bin/python -m scripts.verify_distance_fryer
.venv/bin/python -m scripts.diagnose_transit_similarity
MPLCONFIGDIR=/private/tmp/apma4903-matplotlib .venv/bin/python -m scripts.plot_nyc_eigenvectors
.venv/bin/python -m unittest discover -s tests -q
```
